In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pandas.tseries.offsets import DateOffset
from src.preprocessing import process_store_data
from src.features import attach_store_data, make_features, make_targets

DATA_DIR = '../datasets/rossmann-store-sales'
STORE_FILE = os.path.join(DATA_DIR, 'store.csv')
TRAIN_FILE = os.path.join(DATA_DIR, 'train.csv')
TEST_FILE = os.path.join(DATA_DIR, 'test.csv')

FORECAST_HORIZON = 6*7 # We're forecasting daily for 6 weeks into the future
LAGS = [1, 2, 7, DateOffset(months=1), DateOffset(months=3), DateOffset(months=6)]
DIFFS = [1, DateOffset(months=1), DateOffset(months=3), DateOffset(months=6)]
ROLL_WINDOWS = { 7: 1,                                                              # window: lags
                30: [DateOffset(months=1), DateOffset(months=3), DateOffset(months=6)]}


# NOTE: train_df['Open'] == 0 -> train_df['Sales'] = 0. This happens always


In [2]:
store_df = pd.read_csv(STORE_FILE)
store_df = process_store_data(store_df)

df_train = pd.read_csv(TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)
df_train_store = attach_store_data(df_train, store_df)

# TODO: Predict log-transformed sales?
df_features = make_features(df_train_store, lags=LAGS, roll_windows=ROLL_WINDOWS, diffs=DIFFS)
targets = make_targets(df=df_train[['Date', 'Store', 'Sales']], horizon=FORECAST_HORIZON)

#test_df = pd.read_csv(TEST_FILE, index_col=0, parse_dates=['Date'])
#test_df = features.attach_store_data(test_df, store_df)

C:\Users\m_kal\AppData\Local\Temp\ipykernel_22164\3323829162.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv(TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1)


In [4]:
df_features.shape, targets.shape

((1017209, 64), (1017209, 44))

In [5]:
df_features.head(3)

,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,...,lag_3_months_roll_30_days_median,lag_3_months_roll_30_days_10percentile,lag_3_months_roll_30_days_90percentile,lag_6_months_roll_30_days_mean,lag_6_months_roll_30_days_std,lag_6_months_roll_30_days_skew,lag_6_months_roll_30_days_kurt,lag_6_months_roll_30_days_median,lag_6_months_roll_30_days_10percentile,lag_6_months_roll_30_days_90percentile
0,1,5,2015-07-31,1,1,0,1,c,a,7.147559,...,4139.5,0.0,6228.0,3986.733333,1734.621949,-1.567476,1.647797,4557.5,0.0,5377.6
1,2,5,2015-07-31,1,1,0,1,a,a,6.347389,...,4315.0,0.0,6884.0,4249.933333,2151.852561,-0.751927,-0.144705,4643.0,0.0,6469.2
2,3,5,2015-07-31,1,1,0,1,a,a,9.556126,...,6219.0,0.0,9491.9,5593.166667,2800.879333,-0.752583,0.119621,6000.5,0.0,8312.5


In [6]:
df_features.tail(3)

,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,...,lag_3_months_roll_30_days_median,lag_3_months_roll_30_days_10percentile,lag_3_months_roll_30_days_90percentile,lag_6_months_roll_30_days_mean,lag_6_months_roll_30_days_std,lag_6_months_roll_30_days_skew,lag_6_months_roll_30_days_kurt,lag_6_months_roll_30_days_median,lag_6_months_roll_30_days_10percentile,lag_6_months_roll_30_days_90percentile
1017206,1113,2,2013-01-01,0,0,a,1,a,c,9.133567,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1017207,1114,2,2013-01-01,0,0,a,1,a,c,6.769642,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1017208,1115,2,2013-01-01,0,0,a,1,d,c,8.585039,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
pd.concat([
    df_features.dtypes,
    df_features.isna().sum()/len(df_features),
    df_features.nunique()
], axis=1).sort_values(1, ascending=False).round(2).to_csv('feature_summary.csv')